# F6: first checks against reality (2024–2025), without giving the model the answer

Three questions, each asked of the real data and of the simulation on the same external signals
(experiment E5, `configs/f6_e5.yaml`: `real_v1`, real commodities and S&P 500 of 2024–2025):

1. **Graph out of sample** — does the graph (weights estimated on 2021–2023) describe 2024–2025 better than simple
   models: market beta only, the mean of the sector, a random graph with the same degrees?
2. **Event study** — on the days with a large oil (and copper) move, is the average reaction of every company in
   the simulation the same as in reality?
3. **Lag** — in reality yesterday's move of a parent does not predict today's move of its child. Is our market as
   fast?

Rules of this notebook:

- **No news replay.** The simulation never sees a real stock price of 2024–2025: its only real inputs are four
  commodities and the index. Company news is absent from the simulation and averages out in the real means, which
  is why means are compared, with bootstrap confidence intervals.
- **Every comparison metric is also computed for random graphs** (the same in- and out-degrees, edges rewired,
  weights re-estimated on 2021–2023 exactly like `real_v1`). A metric that a random graph passes equally well
  proves nothing about the graph, and is marked so.
- 2024–2025 is out of sample for the weights. Simulation parameters were calibrated on pilot seeds 1–3 for liquidity
  and stability only (`docs/guides/f6_full_network.md`); nothing here was tuned to the numbers below.

This is a first look for the team. The validation module (`src/marketsim/validation/`, F8) is D1's.

In [ ]:
import copy
import warnings
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from marketsim.data import api
from marketsim.experiments.config import load_config
from marketsim.experiments.run import run_experiment
from marketsim.graph.api import DependencyGraph, Edge, GraphError, load_graph, save_graph
from marketsim.graph.reference import reference_path
from marketsim.graph.weights import estimate_weights

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams["figure.dpi"] = 80
pd.set_option("display.width", 200)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RESULTS = ROOT / "runs" / "f6" / "v1"
WARMUP = 10

GRAPH = load_graph("real_v1")
TICKERS = GRAPH.companies()
returns = api.load_returns_daily()                      # daily log returns 2021-2025: companies, commodities, sp500
external = api.load_external("real_2024_2025")
dates = pd.DatetimeIndex(external["date"])
levels = external.drop(columns="date")
real = returns.loc[dates[1:]]                           # out-of-sample period, 501 days
train = returns.loc["2021-01-01":"2023-12-31"]          # the period the weights were estimated on


def reference_returns(graph):
    # Daily log returns of the reference path R of a graph on the real external signals.
    out = np.log(reference_path(graph, levels, mode="full")).diff().iloc[1:]
    out.index = real.index
    return out[TICKERS]


def market_beta(ticker):
    data = train[["sp500", ticker]].dropna()
    return np.polyfit(data["sp500"], data[ticker], 1)[0]


BETA = pd.Series({t: market_beta(t) for t in TICKERS})
ref_real = reference_returns(GRAPH)
beta_only = pd.DataFrame({t: BETA[t] * real["sp500"] for t in TICKERS})
print(len(real), "days;", len(GRAPH.edges), "edges, of them", sum(e.parent != "sp500" for e in GRAPH.edges),
      "not from the index")

## Random graphs: the control for everything below

Edges that do not start at the index are rewired by random swaps of their children, so every company keeps its
number of parents and every parent its number of children; the index edge of every company stays. Weights are then
re-estimated on 2021–2023 with `estimate_weights`, the function that produced `real_v1` (the check below
reproduces the weights of `real_v1` from its own structure).

In [ ]:
def edge_table(pairs, kinds):
    rows = [(p, c, kinds[p]) for p, c in pairs] + [("sp500", c, "index") for c in TICKERS]
    return pd.DataFrame(rows, columns=["parent", "child", "source"])


def graph_from_structure(pairs, kinds):
    weights = estimate_weights(returns, edge_table(pairs, kinds))
    opt = lambda v: None if np.isnan(v) else float(v)  # noqa: E731
    edges = [Edge(r.parent, r.child, opt(r.w_direct), opt(r.w_full), opt(r.t_stat_direct), opt(r.t_stat_full),
                  r.source, "random control graph") for r in weights.itertuples()]
    return DependencyGraph(dict(GRAPH.nodes), edges, {"name": "random control"}).validate()


REAL_PAIRS = [(e.parent, e.child) for e in GRAPH.edges if e.parent != "sp500"]
KINDS = {p: ("commodity" if GRAPH.nodes[p].kind == "external" else "bea") for p, _ in REAL_PAIRS}


def random_graph(rng, swaps=400):
    pairs = list(REAL_PAIRS)
    for _ in range(swaps):
        i, j = rng.choice(len(pairs), 2, replace=False)
        (p1, c1), (p2, c2) = pairs[i], pairs[j]
        if p1 == c2 or p2 == c1 or (p1, c2) in pairs or (p2, c1) in pairs:
            continue
        pairs[i], pairs[j] = (p1, c2), (p2, c1)
    return graph_from_structure(pairs, KINDS), pairs


rebuilt = graph_from_structure(REAL_PAIRS, KINDS)
check = max(abs(rebuilt.parents(e.child)[e.parent] - e.w_full) for e in GRAPH.edges)
print(f"real structure re-estimated: largest weight difference from real_v1 = {check:.1e}")

rng = np.random.default_rng(7)
RANDOM = []
while len(RANDOM) < 200:
    try:
        RANDOM.append(random_graph(rng))
    except GraphError:      # a rewiring whose feedback loop is unstable
        continue
ref_random = [reference_returns(g) for g, _ in RANDOM]
kept = np.mean([len(set(pairs) & set(REAL_PAIRS)) for _, pairs in RANDOM])
print(f"{len(RANDOM)} random graphs; on average {kept:.1f} of {len(REAL_PAIRS)} real edges survive the rewiring")

## 1. The graph out of sample

Prediction of the daily return of every company in 2024–2025, nothing refitted:

- **R** — the reference path of `real_v1` on the real commodities and index (5 input series, no stock prices);
- **beta only** — market beta (2021–2023) times the S&P 500 return;
- **sector mean** — mean real return of the other companies with the same BEA industry on the same day
  (not available for ODFL and JPM, which are alone in their industry). This model sees real stock prices of the
  same day, which R does not;
- **graph, real parents** — one step of the graph with the real returns of the parents (also sees real prices).

`corr`: correlation with the real return; `R²`: 1 − SSE/SST with the prediction taken as is (it can be negative).

In [ ]:
def fit(pred):
    rows = {}
    for t in TICKERS:
        data = pd.concat([real[t], pred[t]], axis=1).dropna()
        y, p = data.iloc[:, 0].to_numpy(), data.iloc[:, 1].to_numpy()
        if len(y) < 10:
            rows[t] = {"corr": np.nan, "r2": np.nan}
            continue
        rows[t] = {"corr": np.corrcoef(y, p)[0, 1], "r2": 1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)}
    return pd.DataFrame(rows).T


sector = {t: GRAPH.nodes[t].bea_code for t in TICKERS}
peers = {t: [o for o in TICKERS if o != t and sector[o] == sector[t]] for t in TICKERS}
sector_mean = pd.DataFrame({t: real[peers[t]].mean(axis=1) if peers[t] else np.nan for t in TICKERS})
one_step = pd.DataFrame({t: sum(w * real[p] for p, w in GRAPH.parents(t, "full").items()) for t in TICKERS})

models = {"R (real_v1)": ref_real, "beta only": beta_only, "sector mean": sector_mean,
          "graph, real parents": one_step}
fits = pd.concat({name: fit(pred) for name, pred in models.items()}, axis=1)
random_fits = pd.DataFrame([{"median corr": fit(r)["corr"].median(), "median R2": fit(r)["r2"].median(),
                             "mean R2": fit(r)["r2"].mean()} for r in ref_random])
display(fits.round(3))
summary_fit = pd.DataFrame({name: {"median corr": f["corr"].median(), "median R2": f["r2"].median(),
                                   "mean R2": f["r2"].mean()} for name, f in
                            ((n, fits[n]) for n in models)}).T
summary_fit.loc["random graphs: mean"] = random_fits.mean()
summary_fit.loc["random graphs: 95th percentile"] = random_fits.quantile(0.95)
summary_fit.loc["share of random graphs >= real_v1"] = (random_fits >= summary_fit.loc["R (real_v1)"]).mean()
summary_fit.round(3)

In [ ]:
# Is the gain of R over beta-only more than day-to-day luck? Bootstrap over days (blocks of 5 days).
def median_corr_gain(index):
    y = real.iloc[index]
    gain = []
    for t in TICKERS:
        gain.append(np.corrcoef(y[t], ref_real[t].iloc[index])[0, 1] - np.corrcoef(y[t], beta_only[t].iloc[index])[0, 1])
    return np.median(gain), np.mean(gain)


rng = np.random.default_rng(11)
blocks = np.arange(0, len(real) - 4, 5)
draws = []
for _ in range(500):
    starts = rng.choice(blocks, len(blocks))
    draws.append(median_corr_gain(np.concatenate([np.arange(s, s + 5) for s in starts])))
draws = np.array(draws)
point = median_corr_gain(np.arange(len(real)))
print(f"median corr gain of R over beta only: {point[0]:.3f}, 95% CI [{np.quantile(draws[:, 0], 0.025):.3f}, "
      f"{np.quantile(draws[:, 0], 0.975):.3f}]")
print(f"mean corr gain: {point[1]:.3f}, 95% CI [{np.quantile(draws[:, 1], 0.025):.3f}, "
      f"{np.quantile(draws[:, 1], 0.975):.3f}]")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].hist(random_fits["median corr"], bins=25, color="lightgrey", label="200 random graphs")
for name, color in (("R (real_v1)", "C3"), ("beta only", "C0")):
    axes[0].axvline(summary_fit.loc[name, "median corr"], color=color, label=name)
axes[0].set_xlabel("median correlation with real daily returns, 2024-2025")
axes[0].legend()
gain = (fits["R (real_v1)"]["corr"] - fits["beta only"]["corr"]).sort_values()
axes[1].barh(gain.index, gain, color=np.where(gain > 0, "C2", "C3"))
axes[1].set_xlabel("corr(R) - corr(beta only), by company")
plt.tight_layout()
plt.show()

## 2. Simulation runs

Daily close of a run = last two-sided midpoint of the session; simulated daily return = its log change. The first
10 sessions (warmup) are dropped everywhere. Cells of E5: investor mix × herding. If E5 is not there yet, the three
18-company pilot runs are used.

**Random-graph simulations.** The first two random graphs of the list above are also simulated: the agents believe
the random graph (mixed investors, h = 0.2, five seeds each, all other settings as in E5).

In [ ]:
def sim_returns(run_dir):
    book = pd.read_parquet(run_dir / "book.parquet", columns=["t", "asset", "best_bid", "best_ask"])
    book = book.loc[(book.best_bid > 0) & (book.best_ask > 0)]
    book = book.assign(mid=(book.best_bid + book.best_ask) / 2, session=np.floor(book.t).astype(int))
    close = book.groupby(["session", "asset"]).mid.last().unstack().reindex(range(len(dates))).ffill()
    out = np.log(close).diff().iloc[1:]
    out.index, out.columns = real.index, TICKERS
    out.iloc[:WARMUP] = np.nan
    return out


def load_runs(directory, run_ids):
    # Simulated daily returns of several runs, cached next to the runs.
    cache = directory / "daily_returns.parquet"
    if cache.exists():
        table = pd.read_parquet(cache)
        if set(run_ids) <= set(table.run_id):
            return {r: g.drop(columns="run_id") for r, g in table.groupby("run_id") if r in set(run_ids)}
    out = {r: sim_returns(directory / r) for r in run_ids}
    pd.concat([frame.assign(run_id=r) for r, frame in out.items()]).to_parquet(cache)
    return out


CELLS = {}
if (RESULTS / "e5" / "summary.csv").exists() and len(pd.read_csv(RESULTS / "e5" / "summary.csv")) >= 270:
    e5 = pd.read_csv(RESULTS / "e5" / "summary.csv")
    loaded = load_runs(RESULTS / "e5", list(e5.run_id))
    for (mix, h), group in e5.groupby(["agents.composition", "agents.herding.h"]):
        CELLS[f"{mix}, h={h}"] = [loaded[r] for r in group.run_id]
    MAIN = "mixed, h=0.2"
else:
    pilot = pd.read_csv(RESULTS / "pilot" / "summary.csv")
    pilot = pilot.loc[pilot["price_log_change__JPM"].notna() & pilot["price_log_change__NVDA"].notna()]
    CELLS["pilot: mixed, h=0.2"] = list(load_runs(RESULTS / "pilot", list(pilot.run_id)).values())
    MAIN = "pilot: mixed, h=0.2"

# Simulations in which the agents believe a random graph.
RANDOM_SIM = {}
base = load_config(ROOT / "configs" / "f6_e5.yaml").base
for k in (0, 1):
    name = f"random graph {k + 1}"
    graph_dir = RESULTS / "random_graphs" / f"g{k + 1}"
    out_dir = RESULTS / "random_graph_runs" / f"g{k + 1}"
    if not (out_dir / "summary.csv").exists():
        save_graph(RANDOM[k][0], graph_dir)
        config = asdict(base) | {"name": f"f6_random_graph_{k + 1}", "graph": str(graph_dir)}
        spec = {"base": copy.deepcopy(config), "seeds": [4001, 4002, 4003, 4004, 4005], "grid": {}}
        (graph_dir / "experiment.yaml").write_text(yaml.safe_dump(spec, sort_keys=False))
        run_experiment(graph_dir / "experiment.yaml", out_dir, workers=5)   # 5 runs of ~6 minutes
    ids = list(pd.read_csv(out_dir / "summary.csv").run_id)
    RANDOM_SIM[name] = list(load_runs(out_dir, ids).values())
print({name: len(runs) for name, runs in (CELLS | RANDOM_SIM).items()})

## 3. Event study: days with a large commodity move

Events: the 5% of days of 2024–2025 with the largest |Δ ln| of the commodity (after warmup). Every event is signed
by the direction of the commodity, so a positive number means "moved with the commodity".

- **Real**: mean signed return of every company over the events; 95% interval from a bootstrap over events.
- **Simulation**: the same mean on the same days, averaged over the runs of the cell. Because our market may be
  slower, the two-day response (day of the event plus the next day) is shown too.
- **Abnormal** response: the return minus market beta × S&P 500 return of the day, in reality and in the model
  alike. On oil days the index moves too, and every model that knows the betas gets that part right for free.

In [ ]:
def events_of(column, share=0.05):
    move = real[column].iloc[WARMUP:-1]      # the last day has no "next day"
    chosen = move.abs().sort_values(ascending=False).index[: round(share * len(move))]
    return pd.Series(np.sign(move.loc[chosen]), index=chosen).sort_index()


def responses(frame, events, days=1, abnormal=False):
    # Events x companies: signed response over `days` days starting at the event.
    data = frame[TICKERS]
    if abnormal:
        data = data - np.outer(real["sp500"], BETA)
    position = real.index.get_indexer(events.index)
    total = sum(data.to_numpy()[position + d] for d in range(days))
    return total * events.to_numpy()[:, None]


def compare(model, truth, lo, hi):
    return {"corr across companies": np.corrcoef(model, truth)[0, 1],
            "inside real 95% CI": np.mean((model >= lo) & (model <= hi)),
            "same sign": np.mean(np.sign(model) == np.sign(truth)),
            "size ratio": float(model @ truth / (truth @ truth))}


def event_study(column, days=1, abnormal=False):
    events = events_of(column)
    truth_events = responses(real, events, days, abnormal)
    truth = truth_events.mean(axis=0)
    rng = np.random.default_rng(3)
    index = rng.integers(0, len(events), (2000, len(events)))
    boot = truth_events[index].mean(axis=1)
    lo, hi = np.quantile(boot, 0.025, axis=0), np.quantile(boot, 0.975, axis=0)
    means = {"R (real_v1)": responses(ref_real, events, days, abnormal).mean(axis=0),
             "beta only": responses(beta_only, events, days, abnormal).mean(axis=0)}
    for name, runs in (CELLS | RANDOM_SIM).items():
        means["sim: " + name] = np.mean([responses(r, events, days, abnormal).mean(axis=0) for r in runs], axis=0)
    table = pd.DataFrame({name: compare(m, truth, lo, hi) for name, m in means.items()}).T
    control = pd.DataFrame([compare(responses(r, events, days, abnormal).mean(axis=0), truth, lo, hi)
                            for r in ref_random])
    table.loc["R of random graphs: mean"] = control.mean()
    table.loc["R of random graphs: 95th percentile"] = control.quantile(0.95)
    table.loc["share of random R >= R (real_v1)"] = (control >= table.loc["R (real_v1)"]).mean()
    detail = pd.DataFrame({"real": truth, "real lo": lo, "real hi": hi, **means}, index=TICKERS)
    return table, detail, events


oil_table, oil_detail, oil_events = event_study("wti", abnormal=True)
print(f"oil: {len(oil_events)} events, {int((oil_events > 0).sum())} up; mean |move| "
      f"{real.loc[oil_events.index, 'wti'].abs().mean():.3f}; S&P 500 moved with oil by "
      f"{(real.loc[oil_events.index, 'sp500'] * oil_events).mean():+.4f} on average")
oil_table.round(3)

In [ ]:
def plot_events(detail, title, extra=()):
    order = detail["real"].sort_values().index
    x = np.arange(len(order))
    fig, ax = plt.subplots(figsize=(14, 4.2))
    ax.errorbar(x, detail.loc[order, "real"], yerr=[detail.loc[order, "real"] - detail.loc[order, "real lo"],
                detail.loc[order, "real hi"] - detail.loc[order, "real"]], fmt="ko", capsize=3, label="real, 95% CI")
    ax.plot(x, detail.loc[order, "R (real_v1)"], "s", color="C1", label="reference R (real_v1)")
    for name, marker in ((f"sim: {MAIN}", "^"), *extra):
        ax.plot(x, detail.loc[order, name], marker, label=name, alpha=0.8)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_xticks(x, order)
    ax.set_title(title)
    ax.legend(fontsize=8)
    plt.show()


plot_events(oil_detail, "oil events, day 0: mean abnormal response signed by the oil move",
            extra=(("sim: random graph 1", "x"),))

The same comparison for the two-day response, for the raw (not abnormal) response and for copper.

In [ ]:
variants = {
    "oil, day 0, abnormal": oil_table,
    "oil, days 0-1, abnormal": event_study("wti", days=2, abnormal=True)[0],
    "oil, day 0, raw": event_study("wti", days=1, abnormal=False)[0],
    "copper, day 0, abnormal": event_study("copper", days=1, abnormal=True)[0],
    "copper, days 0-1, abnormal": event_study("copper", days=2, abnormal=True)[0],
}
for metric in ("corr across companies", "size ratio", "inside real 95% CI", "same sign"):
    print(metric)
    display(pd.DataFrame({name: table[metric] for name, table in variants.items()}).round(3))

In [ ]:
copper_table, copper_detail, copper_events = event_study("copper", days=2, abnormal=True)
plot_events(copper_detail, "copper events, days 0-1: mean abnormal response signed by the copper move",
            extra=(("sim: random graph 1", "x"),))

### Weeks with an oil move above 5% (the F8 formulation)

Non-overlapping calendar weeks; the response is the weekly return signed by the oil move, abnormal to the index.
Over a week the slowness of our market matters less.

In [ ]:
def weekly(frame):
    out = frame.copy()
    out.iloc[:WARMUP] = np.nan
    return out.groupby(pd.Grouper(freq="W-FRI")).sum(min_count=4)


week_real = weekly(real)
week_oil = week_real["wti"].dropna()
week_events = week_oil[week_oil.abs() > 0.05]
week_sign = np.sign(week_events)


def week_response(frame):
    w = weekly(frame)[TICKERS] - np.outer(week_real["sp500"], BETA)
    return w.loc[week_events.index].to_numpy() * week_sign.to_numpy()[:, None]


truth_weeks = week_response(real)
truth = truth_weeks.mean(axis=0)
rng = np.random.default_rng(5)
boot = truth_weeks[rng.integers(0, len(week_events), (2000, len(week_events)))].mean(axis=1)
lo, hi = np.quantile(boot, 0.025, axis=0), np.quantile(boot, 0.975, axis=0)
rows = {"R (real_v1)": compare(week_response(ref_real).mean(axis=0), truth, lo, hi)}
for name, runs in (CELLS | RANDOM_SIM).items():
    rows["sim: " + name] = compare(np.mean([week_response(r).mean(axis=0) for r in runs], axis=0), truth, lo, hi)
control = pd.DataFrame([compare(week_response(r).mean(axis=0), truth, lo, hi) for r in ref_random])
week_table = pd.DataFrame(rows).T
week_table.loc["R of random graphs: mean"] = control.mean()
week_table.loc["R of random graphs: 95th percentile"] = control.quantile(0.95)
week_table.loc["share of random R >= R (real_v1)"] = (control >= week_table.loc["R (real_v1)"]).mean()
print(f"{len(week_events)} weeks with |oil move| > 5%, {int((week_sign > 0).sum())} up")
week_table.round(3)

## 4. Lag: does yesterday's parent predict today's child?

For every edge of `real_v1` that does not start at the index (33 edges): OLS of the child's return on the parent's
return of the previous day. In the simulation the parent of a commodity edge is the real commodity, the parent of
a company edge is the simulated price of that company; one regression per run and edge.

Second, a pooled view that does not depend on single edges: every company's return on today's and yesterday's
change of its own reference path R (`b0`, `b1`). A market that prices the graph within the day has `b1 ≈ 0`.

**Control:** the same lag regression for the 33 pairs of a random graph (pairs that are *not* edges of `real_v1`),
and for the simulations run on a random graph.

In [ ]:
def ols_t(y, x):
    # OLS with an intercept; returns the coefficients of x and their t-statistics.
    mask = ~(np.isnan(y) | np.isnan(x).any(axis=1))
    y, x = y[mask], np.column_stack([np.ones(mask.sum()), x[mask]])
    beta, *_ = np.linalg.lstsq(x, y, rcond=None)
    resid = y - x @ beta
    cov = resid @ resid / (len(y) - x.shape[1]) * np.linalg.inv(x.T @ x)
    return beta[1:], beta[1:] / np.sqrt(np.diag(cov)[1:])


def lag_tests(frame, pairs):
    # frame: company returns to test; parents that are external come from the real data.
    rows = []
    for parent, child in pairs:
        source = frame[parent] if parent in TICKERS else real[parent]
        b, t = ols_t(frame[child].to_numpy(), source.shift(1).to_numpy()[:, None])
        rows.append({"edge": f"{parent}->{child}", "b_lag": b[0], "t_lag": t[0]})
    return pd.DataFrame(rows).set_index("edge")


def own_reference(frame, reference):
    y = frame[TICKERS].to_numpy().ravel(order="F")
    x0 = reference[TICKERS].to_numpy().ravel(order="F")
    x1 = reference[TICKERS].shift(1).to_numpy().ravel(order="F")
    b, t = ols_t(y, np.column_stack([x0, x1]))
    return {"b0 (same day)": b[0], "b1 (next day)": b[1], "t(b1)": t[1]}


real_after = real.copy()
real_after.iloc[:WARMUP] = np.nan
PLACEBO = [pair for pair in RANDOM[0][1] if pair not in REAL_PAIRS]


def lag_summary(runs, reference, pairs=REAL_PAIRS):
    tests = [lag_tests(r, pairs) for r in runs]
    t = pd.concat([x.t_lag for x in tests], axis=1)
    own = pd.DataFrame([own_reference(r, reference) for r in runs]).mean()
    return {"median t": t.stack().median(), "share |t| > 2": (t.abs() > 2).stack().mean(),
            "share t > 2": (t > 2).stack().mean(), **own}, t.mean(axis=1)


lag_rows, lag_by_edge = {}, {}
lag_rows["real"], lag_by_edge["real"] = lag_summary([real_after], ref_real)
for name, runs in CELLS.items():
    lag_rows["sim: " + name], lag_by_edge["sim: " + name] = lag_summary(runs, ref_real)
for k, (name, runs) in enumerate(RANDOM_SIM.items()):
    lag_rows[f"sim: {name}, its own edges"], _ = lag_summary(runs, ref_random[k], RANDOM[k][1])
lag_rows["real, placebo pairs"], _ = lag_summary([real_after], ref_real, PLACEBO)
lag_rows[f"sim: {MAIN}, placebo pairs"], _ = lag_summary(CELLS[MAIN], ref_real, PLACEBO)
lag_table = pd.DataFrame(lag_rows).T
lag_table.round(3)

In [ ]:
edges = pd.DataFrame({"weight": {f"{e.parent}->{e.child}": e.w_full for e in GRAPH.edges if e.parent != "sp500"},
                      "real t": lag_by_edge["real"], f"sim t ({MAIN})": lag_by_edge[f"sim: {MAIN}"]})
fig, ax = plt.subplots(figsize=(15, 4))
x = np.arange(len(edges))
ax.bar(x - 0.2, edges["real t"], 0.4, label="real 2024-2025")
ax.bar(x + 0.2, edges[f"sim t ({MAIN})"], 0.4, label=f"simulation, {MAIN} (mean over runs)")
for level in (-2, 2):
    ax.axhline(level, color="grey", lw=0.8, ls="--")
ax.set_xticks(x, edges.index, rotation=75, fontsize=8)
ax.set_ylabel("t-statistic of yesterday's parent")
ax.legend()
plt.show()
edges.round(2)

## 5. How close are the simulated returns to the real ones day by day?

Correlation of the simulated daily return of a company with its real return (median over companies, mean over
runs), next to the same number for the reference R. Company news is missing from the simulation, so this can not
be high; the question is only whether the real graph does better than a random one. Two-day returns are added
because the simulated market spreads a move over two days.

In [ ]:
def daily_fit(frames, window=1):
    values = []
    for frame in frames:
        a = real_after[TICKERS].rolling(window).sum().iloc[window - 1::window]
        b = frame[TICKERS].rolling(window).sum().iloc[window - 1::window]
        values.append(a.corrwith(b).median())
    return float(np.mean(values))


rows = {"R (real_v1)": [ref_real], "beta only": [beta_only]}
rows |= {"sim: " + name: runs for name, runs in (CELLS | RANDOM_SIM).items()}
closeness = pd.DataFrame({name: {"1-day returns": daily_fit(frames, 1), "2-day returns": daily_fit(frames, 2),
                                 "5-day returns": daily_fit(frames, 5)} for name, frames in rows.items()}).T
control = pd.DataFrame([{"1-day returns": daily_fit([r], 1), "2-day returns": daily_fit([r], 2),
                         "5-day returns": daily_fit([r], 5)} for r in ref_random])
closeness.loc["R of random graphs: mean"] = control.mean()
closeness.loc["R of random graphs: 95th percentile"] = control.quantile(0.95)
closeness.round(3)

## 6. What these checks say

Filled in after the run.